# Netflix Exploratory Data Analysis (EDA)
## Visualizing Content Trends, Genres, and Demographics

**Objective**: uncover actionable insights about Netflix's catalog evolution, geographic focus, and content strategy.

**Dataset**: Processed Netflix Data (2008-2021)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 7)
plt.rcParams['font.size'] = 12

## 1. Load Data

In [ ]:
df = pd.read_csv('../data/processed/netflix_cleaned.csv')
print(f"Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns")

# Ensure dates are datetime objects
df['date_added_parsed'] = pd.to_datetime(df['date_added_parsed'])
df.head(3)

## 2. Content Strategy: Movies vs TV Shows

In [ ]:
# Distribution of Type
type_counts = df['type'].value_counts()

fig = px.pie(
    values=type_counts.values,
    names=type_counts.index,
    title='Distribution of Content Types',
    color_discrete_sequence=['#b20710', '#221f1f'],
    hole=0.4
)
fig.update_traces(textposition='inside', textinfo='percent+label')
fig.show()

In [ ]:
# Growth over time
growth_df = df.groupby(['year_added', 'type']).size().reset_index(name='count')
growth_df = growth_df[growth_df['year_added'] <= 2021]  # Exclude future/errors

fig = px.area(
    growth_df,
    x='year_added',
    y='count',
    color='type',
    title='Content Added Over Time (2008-2021)',
    color_discrete_map={'Movie': '#b20710', 'TV Show': '#221f1f'}
)
fig.show()

## 3. Genre Analysis

In [ ]:
# Extract first listed genre as primary genre
df['primary_genre'] = df['listed_in'].apply(lambda x: x.split(',')[0].strip())

top_genres = df['primary_genre'].value_counts().head(15)

fig = px.bar(
    x=top_genres.values,
    y=top_genres.index,
    orientation='h',
    title='Top 15 Primary Genres',
    labels={'x': 'Count', 'y': 'Genre'},
    color=top_genres.values,
    color_continuous_scale='Reds'
)
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.show()

## 4. Geographic Analysis

In [ ]:
# Process countries (handle multiple countries)
country_df = df['country'].str.split(', ', expand=True).stack().reset_index(level=1, drop=True)
country_df = country_df[country_df != 'Unknown Country']
country_counts = country_df.value_counts().reset_index(name='count')
country_counts.columns = ['country', 'count']

fig = px.choropleth(
    country_counts,
    locations='country',
    locationmode='country names',
    color='count',
    title='Global Content Distribution',
    color_continuous_scale='Reds',
    range_color=(0, 500)
)
fig.show()

## 5. Content Rating Trends

In [ ]:
# Rating order
rating_order = ['TV-Y', 'TV-Y7', 'TV-G', 'G', 'TV-PG', 'PG', 'PG-13', 'TV-14', 'TV-MA', 'R', 'NC-17']

fig = px.histogram(
    df,
    x='rating',
    color='type',
    barmode='group',
    title='Distribution of Ratings by Content Type',
    category_orders={'rating': rating_order},
    color_discrete_map={'Movie': '#b20710', 'TV Show': '#564d4d'}
)
fig.show()

## 6. Release Timing Analysis

In [ ]:
month_counts = df['month_added'].value_counts().sort_index()
months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

fig = px.bar(
    x=months,
    y=month_counts.values,
    title='Content Additions by Month',
    labels={'x': 'Month', 'y': 'Number of Titles'},
    color=month_counts.values,
    color_continuous_scale='Reds'
)
fig.show()

In [ ]:
heat_data = df.pivot_table(index='month_added', columns='year_added', values='show_id', aggfunc='count', fill_value=0)
heat_data = heat_data[heat_data.columns[(heat_data.columns >= 2015) & (heat_data.columns <= 2021)]]

plt.figure(figsize=(12, 8))
sns.heatmap(heat_data, annot=True, fmt='d', cmap='Reds', yticklabels=months)
plt.title('Content Additions Heatmap (2015-2021)')
plt.ylabel('Month')
plt.xlabel('Year')
plt.show()

## 7. Duration Analysis

In [ ]:
fig = px.histogram(
    df[df['type'] == 'Movie'],
    x='duration_minutes',
    title='Movie Duration Distribution',
    nbins=50,
    color_discrete_sequence=['#b20710']
)
fig.add_vline(x=90, line_dash="dash", line_color="black", annotation_text="90 min")
fig.show()

## 8. Summary & Insights

1. **Content Mix**: Netflix has historically prioritized Movies, but TV Shows have seen rapid growth since 2016.
2. **Global Reach**: While US content dominates, India and UK are significant contributors.
3. **Seasonality**: Content additions tend to peak in Q4 (October-December) and July.
4. **Duration**: Most movies fall in the 90-100 minute range, the industry standard.
5. **Target Audience**: TV-MA and TV-14 are the most common ratings, indicating a focus on mature audiences.